ผู้จัดทำ: กมพันธ์ กันธายอด 6609520116

In [ ]:
# ================================================
# Clustering Benchmark + Cluster Analysis
# KMeans vs DBSCAN vs OPTICS
# ================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN, OPTICS
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# ------------------------------------------------
# 1 Load dataset
# ------------------------------------------------

url = "https://raw.githubusercontent.com/kamonphankanthayod/ml-clustering-credit-card/refs/heads/main/cc_general_mini.csv"

df = pd.read_csv(url)

X = df[["PURCHASES", "PAYMENTS"]]
print(X)

      PURCHASES     PAYMENTS
0         95.40   201.802084
1          0.00  4103.032597
2        773.17   622.066742
3       1499.00     0.000000
4         16.00   678.334763
...         ...          ...
8945     291.12   325.594462
8946     300.00   275.861322
8947     144.40    81.270775
8948       0.00    52.549959
8949    1093.25    63.165404

[8950 rows x 2 columns]


In [ ]:
# ------------------------------------------------
# 2 Standardize data
# ------------------------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled)

[[-0.42489974 -0.52897879]
 [-0.46955188  0.81864213]
 [-0.10766823 -0.38380474]
 ...
 [-0.40196519 -0.5706145 ]
 [-0.46955188 -0.58053567]
 [ 0.04214581 -0.57686873]]


In [17]:
# ------------------------------------------------
# 3 Hyperparameter grids
# ------------------------------------------------

# Note: The 'divide by zero' RuntimeWarning mentioned in your query
# comes from the OPTICS algorithm, not DBSCAN, and is discussed in the
# accompanying text response.

kmeans_grid = range(2,9)

dbscan_eps = [0.3,0.4,0.5,0.7,1.0]
dbscan_min_samples = [3,4,5,6]

optics_min_samples = [3,4,5,6,7]

results = []

# ------------------------------------------------
# 4 KMeans grid search
# ------------------------------------------------

for k in kmeans_grid:

    model = KMeans(n_clusters=k, random_state=42, n_init=10)

    labels = model.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, labels)
    dbi = davies_bouldin_score(X_scaled, labels)

    n_clusters = len(set(labels))

    results.append(["KMeans", f"k={k}", n_clusters, sil, dbi, labels])

# ------------------------------------------------
# 5 DBSCAN grid search
# ------------------------------------------------

for eps in dbscan_eps:
    for m in dbscan_min_samples:

        model = DBSCAN(eps=eps, min_samples=m)

        labels = model.fit_predict(X_scaled)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        if n_clusters > 1:

            sil = silhouette_score(X_scaled, labels)
            dbi = davies_bouldin_score(X_scaled, labels)

            results.append(["DBSCAN", f"eps={eps},min_samples={m}", n_clusters, sil, dbi, labels])

# ------------------------------------------------
# 6 OPTICS grid search
# ------------------------------------------------

for m in optics_min_samples:

    model = OPTICS(min_samples=m)

    labels = model.fit_predict(X_scaled)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    if n_clusters > 1:

        sil = silhouette_score(X_scaled, labels)
        dbi = davies_bouldin_score(X_scaled, labels)

        results.append(["OPTICS", f"min_samples={m}", n_clusters, sil, dbi, labels])

/usr/local/lib/python3.12/dist-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/usr/local/lib/python3.12/dist-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/usr/local/lib/python3.12/dist-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/usr/local/lib/python3.12/dist-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/usr/local/lib/python3.12/dist-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


In [18]:
# ------------------------------------------------
# 7 Ranking models
# ------------------------------------------------

result_df = pd.DataFrame(
    results,
    columns=["Algorithm","Parameters","Clusters","Silhouette","DBI","Labels"]
)

result_df = result_df.sort_values(
    by=["Silhouette","DBI"],
    ascending=[False,True]
).reset_index(drop=True)

result_df["Rank"] = result_df.index + 1

print("\n========== Clustering Model Ranking ==========\n")

print(result_df[["Rank","Algorithm","Parameters","Clusters","Silhouette","DBI"]])


========== Clustering Model Ranking ==========

    Rank Algorithm             Parameters  Clusters  Silhouette       DBI
0      1    DBSCAN  eps=1.0,min_samples=5         2    0.914049  1.297193
1      2    DBSCAN  eps=1.0,min_samples=4         3    0.908701  1.720452
2      3    DBSCAN  eps=1.0,min_samples=3         5    0.885216  1.346819
3      4    DBSCAN  eps=0.7,min_samples=5         2    0.857814  1.527551
4      5    DBSCAN  eps=0.7,min_samples=4         2    0.856830  1.287516
5      6    DBSCAN  eps=0.7,min_samples=3         5    0.854349  1.832872
6      7    DBSCAN  eps=0.5,min_samples=6         2    0.843252  0.844800
7      8    KMeans                    k=2         2    0.841423  0.746141
8      9    DBSCAN  eps=0.3,min_samples=6         3    0.798757  1.592785
9     10    DBSCAN  eps=0.4,min_samples=6         4    0.796151  1.356548
10    11    DBSCAN  eps=0.4,min_samples=5         5    0.768246  1.310108
11    12    DBSCAN  eps=0.5,min_samples=5         2    0.765516

In [20]:
print(len(results))

30


-------------------------------------------------------

พบว่าใช้ขั้นตอนวิธี DBSCAN โดยกำหนดค่า eps = 1.0, min_samples = 5 ได้กลุ่ม 2 กลุ่ม ได้ค่าประสิทธิภาพดีที่สุด

----------------------------------------------------